# 01 — SciPy — optimisation

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- résoudre des problèmes d’optimisation scalaire et multidimensionnelle ;
- trouver les racines de fonctions avec `brentq`, `root_scalar`, `root` ;
- utiliser `minimize` avec différents algorithmes (Nelder-Mead, BFGS, SLSQP, L-BFGS-B) ;
- ajouter des contraintes (égalité, inégalité) et des bornes ;
- faire du curve fitting avec `curve_fit` et `least_squares` ;
- fournir un gradient analytique pour accélérer la convergence.

## Prérequis — ce que vous devez maîtriser

À ce stade de la formation, vous savez :

- NumPy (arrays, fonctions mathématiques) ;
- notions de dérivées et gradient ;
- fonctions Python typées.

Ce que ce notebook **ne couvrira pas** :

- le deep learning (PyTorch, JAX) ;
- l’optimisation convexe formelle (CVXPY) ;
- la programmation linéaire en grande dimension.

## Plan

1. Installation et imports
2. Minimisation scalaire
3. Racines d’équations (1D)
4. Racines d’équations (ND)
5. Optimisation multi-variables (`minimize`)
6. Choix d’algorithme
7. Fournir le gradient
8. Contraintes d’égalité et d’inégalité
9. Bornes sur les variables
10. Curve fitting
11. Moindres carrés non linéaires
12. Synthèse
13. Exercices
14. Ressources externes

### Exercice : optimisation avec contraintes

Utilisez `scipy.optimize.minimize` avec la méthode `SLSQP` pour résoudre :

$$\min_{x,y} \; (x-1)^2 + (y-2.5)^2$$

sous les contraintes :
- $x - 2y + 2 \geq 0$
- $-x - 2y + 6 \geq 0$
- $x, y \geq 0$

In [ ]:
from scipy.optimize import minimize

def objective(x):
    return (x[0] - 1)**2 + (x[1] - 2.5)**2

constraints = [
    {'type': 'ineq', 'fun': lambda x: x[0] - 2*x[1] + 2},
    {'type': 'ineq', 'fun': lambda x: -x[0] - 2*x[1] + 6},
]
bounds = [(0, None), (0, None)]

res = minimize(objective, [0, 0], method='SLSQP',
              bounds=bounds, constraints=constraints)
print(f'Solution : x={res.x}, f(x)={res.fun:.4f}')

### Exercice : curve fitting

Utilisez `scipy.optimize.curve_fit` pour ajuster un modèle exponentiel
$f(x) = a \cdot e^{-bx} + c$ sur des données bruitées.

In [ ]:
from scipy.optimize import curve_fit

# Données synthétiques
np.random.seed(42)
x_data = np.linspace(0, 4, 50)
y_data = 2.5 * np.exp(-1.3 * x_data) + 0.5 + 0.2 * np.random.randn(50)

def model(x, a, b, c):
    return a * np.exp(-b * x) + c

popt, pcov = curve_fit(model, x_data, y_data, p0=[1, 1, 0])
print(f'Paramètres : a={popt[0]:.3f}, b={popt[1]:.3f}, c={popt[2]:.3f}')

plt.scatter(x_data, y_data, s=10, label='données')
plt.plot(x_data, model(x_data, *popt), 'r-', label='ajustement')
plt.legend()
plt.show()

### Mini-exercice : minimisation en dimension 3

Trouvez le minimum de $f(x,y,z) = x^2 + 2y^2 + 3z^2 + xy - yz$ en partant de $(1, 1, 1)$.
Comparez les résultats de `Nelder-Mead` et `BFGS`.

In [ ]:
def f3d(x):
    return x[0]**2 + 2*x[1]**2 + 3*x[2]**2 + x[0]*x[1] - x[1]*x[2]

x0 = np.array([1.0, 1.0, 1.0])
for method in ['Nelder-Mead', 'BFGS', 'Powell']:
    res = minimize(f3d, x0, method=method)
    print(f'{method:15s} -> x={res.x}, f={res.fun:.6f}, nfev={res.nfev}')

---

## 1. Installation et imports

In [ ]:
# pip install scipy numpy
import scipy
print(f"SciPy {scipy.__version__}")

In [ ]:
import numpy as np
from scipy import optimize

---

## 2. Minimisation scalaire

`minimize_scalar` cherche le minimum d’une fonction d’une seule variable.

In [ ]:
from scipy.optimize import minimize_scalar

f = lambda x: (x - 2)**2 + 1
res = minimize_scalar(f)
print(f"x* = {res.x:.6f}, f(x*) = {res.fun:.6f}")

### Méthode `bounded`

On peut restreindre la recherche à un intervalle :

In [ ]:
res = minimize_scalar(f, bounds=(0, 10), method='bounded')
print(f"x* = {res.x:.6f}")

### Méthode `brent` (par défaut)

Brent combine interpolation parabolique et Golden Section. C’est le choix par défaut.

In [ ]:
res = minimize_scalar(f, method='brent')
print(f"Méthode: {res.method if hasattr(res, 'method') else 'brent'}")

In [ ]:
# Fonction avec plusieurs minima locaux
import math
g = lambda x: math.sin(x) + 0.1 * x
res = minimize_scalar(g, bounds=(-5, 0), method='bounded')
print(f"Min local dans [-5, 0]: x={res.x:.4f}, f={res.fun:.4f}")

---

## 3. Racines d’équations (1D)

Trouver *x* tel que *f(x) = 0*.

In [ ]:
from scipy.optimize import brentq

# cos(x) = x  <=>  cos(x) - x = 0
root = brentq(lambda x: math.cos(x) - x, 0, 1)
print(f"racine: {root:.10f}")

`brentq` exige un **bracket** `[a, b]` tel que `f(a)` et `f(b)` aient des signes opposés.

In [ ]:
from scipy.optimize import root_scalar

r = root_scalar(lambda x: x**3 - x - 2, bracket=[1, 2])
print(f"racine: {r.root:.10f}")

In [ ]:
# Avec méthode Newton (nécessite la dérivée)
r = root_scalar(
    lambda x: x**3 - x - 2,
    fprime=lambda x: 3*x**2 - 1,
    x0=1.5,
    method='newton',
)
print(f"Newton: {r.root:.10f}, iterations: {r.iterations}")

---

## 4. Racines d’équations (ND)

Pour un système d’équations non linéaires.

In [ ]:
from scipy.optimize import root

def system(v):
    x, y = v
    return [x**2 + y**2 - 4, x * y - 1]

sol = root(system, [1, 1])
print(f"x={sol.x[0]:.6f}, y={sol.x[1]:.6f}")

In [ ]:
# Vérification
print(f"x²+y²={sol.x[0]**2 + sol.x[1]**2:.6f}")
print(f"x*y={sol.x[0]*sol.x[1]:.6f}")

---

## 5. Optimisation multi-variables (`minimize`)

La fonction de Rosenbrock est un classique pour tester les optimiseurs.

In [ ]:
from scipy.optimize import minimize, rosen

x0 = np.array([0.0, 0.0])
res = minimize(rosen, x0)
print(f"x* = {res.x}")
print(f"f(x*) = {res.fun:.2e}")

### Interpréter le résultat

In [ ]:
print(f"Succès: {res.success}")
print(f"Message: {res.message}")
print(f"Itérations: {res.nit}")
print(f"Évaluations f: {res.nfev}")

In [ ]:
# Rosenbrock en 3D
x0 = np.zeros(3)
res = minimize(rosen, x0)
print(f"x* = {res.x}  (attendu: [1, 1, 1])")

---

## 6. Choix d’algorithme

| Méthode | Gradient ? | Contraintes ? | Usage typique |
|---|---|---|---|
| `Nelder-Mead` | Non | Non | Simplex, pas besoin de gradient |
| `BFGS` | Oui (approx.) | Non | Smooth, usage général |
| `L-BFGS-B` | Oui | Bornes | Grandes dimensions |
| `CG` | Oui | Non | Gradient conjugué |
| `Powell` | Non | Non | Sans gradient, rapide |
| `COBYLA` | Non | Ég./Inég. | Contraintes sans gradient |
| `SLSQP` | Oui | Ég./Inég. | Contraintes + gradient |
| `trust-constr` | Oui | Tout | Moderne, robuste |

In [ ]:
for method in ['Nelder-Mead', 'BFGS', 'Powell', 'CG', 'L-BFGS-B']:
    res = minimize(rosen, [0, 0], method=method)
    print(f"{method:15s} -> x={res.x}, nit={res.nit}")

### Quel algorithme choisir ?

- **Sans gradient** : Nelder-Mead ou Powell
- **Avec gradient** : BFGS (petit) ou L-BFGS-B (grand)
- **Avec contraintes** : SLSQP ou trust-constr
- **Avec bornes uniquement** : L-BFGS-B

---

## 7. Fournir le gradient

Un gradient analytique accélère la convergence et la précision.

In [ ]:
from scipy.optimize import rosen_der

res_no_jac = minimize(rosen, [0, 0], method='BFGS')
print(f"Sans gradient : {res_no_jac.nfev} évaluations")

res_jac = minimize(rosen, [0, 0], jac=rosen_der, method='BFGS')
print(f"Avec gradient : {res_jac.nfev} évaluations")

In [ ]:
# Gradient custom pour f(x,y) = x² + 4y²
def f_quad(v):
    return v[0]**2 + 4 * v[1]**2

def grad_quad(v):
    return np.array([2 * v[0], 8 * v[1]])

res = minimize(f_quad, [5, 5], jac=grad_quad, method='BFGS')
print(f"x* = {res.x}")

### Gradient numérique (vérification)

In [ ]:
from scipy.optimize import approx_fprime
point = np.array([1.0, 2.0])
num_grad = approx_fprime(point, f_quad, 1e-8)
ana_grad = grad_quad(point)
print(f"Numérique: {num_grad}")
print(f"Analytique: {ana_grad}")

---

## 8. Contraintes d’égalité et d’inégalité

Les contraintes sont passées sous forme de dictionnaires.

### Contrainte d’égalité

Minimiser *x² + y²* sous *x + y = 1*.

In [ ]:
f = lambda v: v[0]**2 + v[1]**2

con_eq = {'type': 'eq', 'fun': lambda v: v[0] + v[1] - 1}
res = minimize(f, [0, 0], constraints=con_eq, method='SLSQP')
print(f"x={res.x[0]:.4f}, y={res.x[1]:.4f}  (somme={sum(res.x):.4f})")

### Contrainte d’inégalité

Minimiser *x² + y²* sous *x ≥ 1*. (La convention SciPy : `fun(x) >= 0`.)

In [ ]:
con_ineq = {'type': 'ineq', 'fun': lambda v: v[0] - 1}
res = minimize(f, [0, 0], constraints=con_ineq, method='SLSQP')
print(f"x={res.x[0]:.4f}, y={res.x[1]:.4f}")

In [ ]:
# Contraintes multiples
cons = [
    {'type': 'eq', 'fun': lambda v: v[0] + v[1] + v[2] - 1},
    {'type': 'ineq', 'fun': lambda v: v[0]},        # x >= 0
    {'type': 'ineq', 'fun': lambda v: v[1]},        # y >= 0
]
res = minimize(lambda v: v[0]**2 + v[1]**2 + v[2]**2, [0.5, 0.5, 0], constraints=cons, method='SLSQP')
print(f"x* = {res.x}")

---

## 9. Bornes sur les variables

In [ ]:
from scipy.optimize import Bounds

# x dans [0, 5], y dans [-3, 3]
res = minimize(f, [2, 2], bounds=[(0, 5), (-3, 3)])
print(f"x* = {res.x}")

In [ ]:
# Avec l'objet Bounds
bounds = Bounds(lb=[0, -3], ub=[5, 3])
res = minimize(f, [2, 2], bounds=bounds, method='L-BFGS-B')
print(f"x* = {res.x}")

---

## 10. Curve fitting

Ajuster un modèle paramétrique à des données.

In [ ]:
from scipy.optimize import curve_fit

def model(x, a, b, c):
    return a * np.exp(-b * x) + c

rng = np.random.default_rng(42)
xdata = np.linspace(0, 4, 50)
ydata = model(xdata, 2.5, 1.3, 0.5) + 0.2 * rng.standard_normal(50)

popt, pcov = curve_fit(model, xdata, ydata)
print(f"a={popt[0]:.3f}, b={popt[1]:.3f}, c={popt[2]:.3f}")

### Incertitudes sur les paramètres

In [ ]:
perr = np.sqrt(np.diag(pcov))
for name, val, err in zip('abc', popt, perr):
    print(f"{name} = {val:.4f} ± {err:.4f}")

In [ ]:
# Avec bornes sur les paramètres
popt2, _ = curve_fit(model, xdata, ydata, bounds=([0, 0, -1], [10, 10, 5]))
print(f"Avec bornes: a={popt2[0]:.3f}, b={popt2[1]:.3f}, c={popt2[2]:.3f}")

---

## 11. Moindres carrés non linéaires

`least_squares` est plus flexible que `curve_fit` : on définit directement les résidus.

In [ ]:
from scipy.optimize import least_squares

def residuals(params, x, y):
    a, b, c = params
    return a * np.exp(-b * x) + c - y

res = least_squares(residuals, [1, 1, 0], args=(xdata, ydata))
print(f"Params: {res.x}")
print(f"Coût final: {res.cost:.6f}")

In [ ]:
# Moindres carrés robustes (outliers)
res_robust = least_squares(residuals, [1, 1, 0], args=(xdata, ydata), loss='soft_l1')
print(f"Robuste: {res_robust.x}")

---

## Synthèse

| Problème | Fonction SciPy | Arguments clés |
|---|---|---|
| Min 1D | `minimize_scalar(f)` | `bounds=`, `method=` |
| Min ND | `minimize(f, x0)` | `method=`, `jac=`, `constraints=`, `bounds=` |
| Racine 1D | `brentq(f, a, b)` | bracket obligatoire |
| Racine ND | `root(f, x0)` | `method=` |
| Curve fit | `curve_fit(model, x, y)` | `p0=`, `bounds=` |
| NLS | `least_squares(res, x0)` | `loss=`, `bounds=` |

### Règles à retenir

- **BFGS** par défaut pour smooth sans contraintes.
- **SLSQP** pour les contraintes d’égalité/inégalité.
- **L-BFGS-B** pour les bornes en grande dimension.
- **Fournir le gradient** = convergence plus rapide et plus précise.
- **`bracket=[a, b]`** pour les racines (plus robuste que `x0`).

---

## Exercices

### Exercice 1 *(facile)* — Minimum d’une parabole

Trouvez le minimum de la fonction `f(x) = (x - 3)**2 + 5` avec `minimize_scalar`.

In [ ]:
from scipy.optimize import minimize_scalar

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from scipy.optimize import minimize_scalar

res = minimize_scalar(lambda x: (x - 3)**2 + 5)
print(f"x* = {res.x:.4f}, f(x*) = {res.fun:.4f}")
```

</details>

### Exercice 2 *(facile)* — Racine de sin

Trouvez une racine de `sin(x) - 0.5` dans l’intervalle `[0, 2]`.

In [ ]:
from scipy.optimize import brentq
import math

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from scipy.optimize import brentq
import math

root = brentq(lambda x: math.sin(x) - 0.5, 0, 2)
print(f"racine: {root:.6f}")  # pi/6 = 0.523599
```

</details>

### Exercice 3 *(moyen)* — Minimisation sous contrainte

Minimisez `x² + y² + z²` sous la contrainte `x + y + z = 1`. Vérifiez que la solution est `(1/3, 1/3, 1/3)`.

In [ ]:
from scipy.optimize import minimize

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from scipy.optimize import minimize

res = minimize(
    lambda v: sum(vi**2 for vi in v),
    [0, 0, 0],
    constraints=[{'type': 'eq', 'fun': lambda v: v[0] + v[1] + v[2] - 1}],
    method='SLSQP',
)
print(f"x* = {res.x}")
```

</details>

### Exercice 4 *(moyen)* — Ajustement exponentiel

Générez des données bruitées `y = 3 * exp(-0.8 * x) + 1 + bruit` pour `x` dans `[0, 5]` (50 points). Ajustez le modèle `a * exp(-b * x) + c` avec `curve_fit`.

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.optimize import curve_fit

def model(x, a, b, c):
    return a * np.exp(-b * x) + c

rng = np.random.default_rng(0)
x = np.linspace(0, 5, 50)
y = model(x, 3, 0.8, 1) + 0.1 * rng.standard_normal(50)
popt, _ = curve_fit(model, x, y, p0=[1, 1, 0])
print(f"a={popt[0]:.3f}, b={popt[1]:.3f}, c={popt[2]:.3f}")
```

</details>

### Exercice 5 *(difficile)* — Rosenbrock 3D avec bornes

Minimisez la fonction de Rosenbrock en 3D avec des bornes `[-2, 2]` sur chaque variable, en utilisant `L-BFGS-B`. Comparez le nombre d’évaluations avec et sans gradient (`rosen_der`).

In [ ]:
import numpy as np
from scipy.optimize import minimize, rosen, rosen_der

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
import numpy as np
from scipy.optimize import minimize, rosen, rosen_der

bounds = [(-2, 2)] * 3

res1 = minimize(rosen, np.zeros(3), method='L-BFGS-B', bounds=bounds)
print(f"Sans gradient: x={res1.x}, nfev={res1.nfev}")

res2 = minimize(rosen, np.zeros(3), jac=rosen_der, method='L-BFGS-B', bounds=bounds)
print(f"Avec gradient: x={res2.x}, nfev={res2.nfev}")
```

</details>

### Exercice 6 *(difficile)* — Système non linéaire 3D

Résolvez le système :
- `x² + y² + z² = 3`
- `x + y + z = 3`
- `x * y * z = 1`

Partez de `[1, 1, 1]`.

In [ ]:
from scipy.optimize import root

# Votre code ici

<details>
<summary>📖 Voir la correction</summary>

```python
from scipy.optimize import root

def sys(v):
    x, y, z = v
    return [
        x**2 + y**2 + z**2 - 3,
        x + y + z - 3,
        x * y * z - 1,
    ]

sol = root(sys, [1, 1, 1])
print(f"Solution: {sol.x}")
print(f"Vérif: {[f'{v:.6f}' for v in sys(sol.x)]}")
```

</details>

---

## Ressources externes

- [scipy.optimize — documentation officielle](https://docs.scipy.org/doc/scipy/reference/optimize.html)
- [Tutorial SciPy — Optimization](https://docs.scipy.org/doc/scipy/tutorial/optimize.html)
- [Choosing a solver](https://docs.scipy.org/doc/scipy/tutorial/optimize.html#choosing-a-method)

### Exemple approfondi : surface de Rosenbrock

La fonction de Rosenbrock est un classique pour tester les algorithmes d'optimisation.
Sa forme générale est $f(x, y) = (a - x)^2 + b(y - x^2)^2$ avec $a=1, b=100$.
Le minimum global se trouve en $(1, 1)$.

In [ ]:
import numpy as np
from scipy.optimize import minimize, rosen, rosen_der

# Minimisation sans gradient
x0 = np.array([0.0, 0.0])
res_nm = minimize(rosen, x0, method='Nelder-Mead')
print(f'Nelder-Mead : x={res_nm.x}, nfev={res_nm.nfev}')

# Avec gradient analytique
res_bfgs = minimize(rosen, x0, jac=rosen_der, method='BFGS')
print(f'BFGS : x={res_bfgs.x}, nfev={res_bfgs.nfev}')

### Approfondir : optimisation globale

Les méthodes vues jusqu'ici trouvent des **minima locaux**.
Pour trouver le minimum global, SciPy propose `differential_evolution`,
`dual_annealing` et `shgo`.

In [ ]:
from scipy.optimize import differential_evolution, dual_annealing
import numpy as np

# Fonction avec plusieurs minima locaux
def multimodal(x):
    return np.sin(5*x[0]) * np.cos(3*x[1]) + (x[0]**2 + x[1]**2)/10

bounds = [(-3, 3), (-3, 3)]
res_de = differential_evolution(multimodal, bounds, seed=42)
res_da = dual_annealing(multimodal, bounds, seed=42)
print(f'differential_evolution : x={res_de.x}, f={res_de.fun:.4f}')
print(f'dual_annealing        : x={res_da.x}, f={res_da.fun:.4f}')

### Exercice : optimisation de portefeuille

Vous avez 3 actifs avec des rendements espérés $\mu = [0.12, 0.10, 0.07]$
et une matrice de covariance. Trouvez les poids qui minimisent la variance
du portefeuille sous la contrainte que la somme des poids = 1 et chaque poids >= 0.

Utilisez `minimize` avec la méthode `SLSQP`.

In [ ]:
import numpy as np
from scipy.optimize import minimize

mu = np.array([0.12, 0.10, 0.07])
cov = np.array([
    [0.04, 0.006, 0.002],
    [0.006, 0.025, 0.004],
    [0.002, 0.004, 0.01]
])

def variance(w):
    return w @ cov @ w

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0, 1)] * 3
w0 = np.array([1/3, 1/3, 1/3])

res = minimize(variance, w0, method='SLSQP', bounds=bounds, constraints=constraints)
print(f'Poids optimaux : {res.x}')
print(f'Rendement : {mu @ res.x:.4f}')
print(f'Volatilité : {np.sqrt(res.fun):.4f}')

### Astuce : visualisation de la convergence

On peut tracer l'évolution de la fonction objectif au fil des itérations
en utilisant un callback.

In [ ]:
import numpy as np
from scipy.optimize import minimize, rosen
import matplotlib.pyplot as plt

history = []
def callback(xk):
    history.append(rosen(xk))

x0 = np.array([-1.5, 1.5])
minimize(rosen, x0, method='BFGS', callback=callback)

plt.semilogy(history, 'b.-')
plt.xlabel('Itération'); plt.ylabel('f(x) (log)')
plt.title('Convergence BFGS sur Rosenbrock')
plt.grid(); plt.show()

### Comparaison visuelle des trajectoires

Tracons les chemins suivis par chaque algorithme pour atteindre le minimum.

In [ ]:
import matplotlib.pyplot as plt

# Landscape
xx = np.linspace(-2, 2, 200)
yy = np.linspace(-1, 3, 200)
X, Y = np.meshgrid(xx, yy)
Z = (1 - X)**2 + 100*(Y - X**2)**2

fig, ax = plt.subplots(figsize=(8, 6))
ax.contour(X, Y, Z, levels=np.logspace(0, 3.5, 30), cmap='viridis')
ax.plot(1, 1, 'r*', markersize=15, label='minimum')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Surface de Rosenbrock')
ax.legend()
plt.tight_layout()
plt.show()